# Calidad de Datos — Clientes SARLAFT / PEP (versión `sarlaf.sql`)

Ejecuta [`sarlaf/sarlaf.sql`](sarlaf.sql) completo desde el notebook, con el **mismo SQL literal**
del archivo (solo se parametrizan el periodo y la tabla PEP, que en el original están quemados) y
añade una sección de **diagnóstico de la regla de CONSISTENCIA**, que hoy devuelve `NULL` para
persona jurídica.

Sucede a [`../notebooks/dq_clientes_sarlaft_pep_runner.ipynb`](../notebooks/dq_clientes_sarlaft_pep_runner.ipynb):
esta versión del SQL agrega la dimensión **consistencia** (comparación contra la lista Pirani),
las llaves normalizadas de unicidad (`rb_norm`) y el bloque de salida en **formato SFC**.

**Qué escribe en la base:** nada permanente. Los tres bloques crean solo **tablas TEMP de sesión**
(el `CREATE TABLE tmp_reporte_sfc_calidad_datos` del SQL original se ejecuta aquí como
`CREATE TEMP TABLE`; ver hallazgo #2 al final). Del resultado solo salen **agregados**: ningún dato
personal se escribe en `reports/`.

**Requiere red/VPN corporativa** para llegar a Redshift.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import redshift_connector

pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

## Conexión a Redshift

Mismo patrón que los notebooks de `notebooks/`: las credenciales se leen de
`notebooks/credenciales_local.py` (fuera de git). Si no existe, se piden por consola y no quedan
guardadas.

In [2]:
# credenciales_local.py vive en notebooks/; se agrega esa carpeta al path para reusarlo.
REPO = Path.cwd().parent if Path.cwd().name == 'sarlaf' else Path.cwd()
sys.path.insert(0, str(REPO / 'notebooks'))

try:
    from credenciales_local import (
        DEFAULT_HOST, DEFAULT_PORT, DEFAULT_DATABASE, DEFAULT_USER, DEFAULT_PASSWORD
    )
    print('Credenciales cargadas desde notebooks/credenciales_local.py (fuera de git).')
except ImportError:
    import getpass
    print('No se encontró credenciales_local.py — ingresa las credenciales:')
    DEFAULT_HOST = input('host: ').strip()
    DEFAULT_PORT = input('puerto: ').strip()
    DEFAULT_DATABASE = input('base de datos: ').strip()
    DEFAULT_USER = input('usuario: ').strip()
    DEFAULT_PASSWORD = getpass.getpass('contraseña: ')

Credenciales cargadas desde notebooks/credenciales_local.py (fuera de git).


In [3]:
conn = redshift_connector.connect(
    host=DEFAULT_HOST,
    port=int(DEFAULT_PORT),
    database=DEFAULT_DATABASE,
    user=DEFAULT_USER,
    password=DEFAULT_PASSWORD,
)
conn.autocommit = True  # las TEMP TABLE viven solo en esta sesión; nada permanente
cursor = conn.cursor()
print('Conectado. Este notebook solo crea tablas TEMPORALES de sesión.')

Conectado. Este notebook solo crea tablas TEMPORALES de sesión.


In [4]:
def run_query(sql):
    'Ejecuta un SELECT y devuelve un DataFrame.'
    cursor.execute(sql)
    return cursor.fetch_dataframe()


def run_statement(sql):
    'Ejecuta una sentencia sin resultado (DROP / CREATE TEMP TABLE).'
    cursor.execute(sql)

## Parámetros: periodo y tabla PEP (autodetectados)

`sarlaf.sql` trae quemados el periodo `'202606'` y la tabla `clientes_pirani_202606`. Aquí se
detectan solos —último periodo con datos en `fact_vigentes_total_historico` y última tabla
`clientes_pirani_*` del esquema—, alineado con el mandato del proyecto de no dejar periodos
quemados. `PERIODO_MANUAL` / `TABLA_PEP_MANUAL` permiten forzarlos.

In [5]:
PERIODO_MANUAL = None    # ej. '202606' para forzar un periodo puntual
TABLA_PEP_MANUAL = None  # ej. 'clientes_pirani_202606' para forzar la tabla PEP

df_per = run_query('''
select max(accountable_period) as periodo
from co_sandbox_datos.fact_vigentes_total_historico;
''')
PERIODO = str(PERIODO_MANUAL or df_per['periodo'].iloc[0]).strip()
assert len(PERIODO) == 6 and PERIODO.isdigit(), f'Periodo inesperado: {PERIODO!r}'

df_pep = run_query('''
select table_name
from information_schema.tables
where table_schema = 'co_sandbox_datos'
  and table_name like 'clientes_pirani%'
order by table_name desc;
''')
if TABLA_PEP_MANUAL:
    TABLA_PEP = TABLA_PEP_MANUAL
elif len(df_pep):
    # se prefiere la tabla del mismo periodo; si no existe, la más reciente
    candidatas = list(df_pep['table_name'])
    TABLA_PEP = next((t for t in candidatas if t.endswith(PERIODO)), candidatas[0])
else:
    raise RuntimeError('No se encontró ninguna tabla clientes_pirani_* en co_sandbox_datos.')

print(f'PERIODO   = {PERIODO}')
print(f'TABLA_PEP = {TABLA_PEP}')
print('Tablas PEP disponibles:', list(df_pep['table_name']))

PERIODO   = 202606
TABLA_PEP = clientes_pirani_202606
Tablas PEP disponibles: ['clientes_pirani_202606', 'clientes_pirani_202605', 'clientes_pirani_202604']


## Bloque A — Reporte base (`tmp_reporte_base`)

Tomadores con póliza vigente en el periodo, enriquecidos con la **mejor fuente disponible por
atributo** (prioridad SCT → Formulario SARLAFT → iAxis → AS400), más la lista PEP de Pirani
(`clientes_pep`, `ingresos_pirani`, `egresos_pirani` — estas dos últimas son la base de la
dimensión *consistencia*).

SQL idéntico al de `sarlaf.sql`, con periodo y tabla PEP parametrizados.

In [6]:
# Bloque A tal cual el SQL original, con periodo y tabla PEP parametrizados.
BLOQUE_A = f'''
CREATE TEMP TABLE tmp_reporte_base AS
WITH vigentes_base AS (
    SELECT
        v.policy_holder,
        v.accountable_period,
        ROW_NUMBER() OVER (
            PARTITION BY v.policy_holder
            ORDER BY v.certificate_start_date DESC
        ) AS rn
    FROM co_sandbox_datos.fact_vigentes_total_historico v
    WHERE v.accountable_period = '{PERIODO}'
),
vigentes_filtrado AS (
    SELECT
        policy_holder
    FROM vigentes_base
    WHERE rn = 1
),
per_personas_base AS (
    SELECT
        p.sperson,
        p.ctipide,
        CAST(p.nnumide AS VARCHAR(50)) AS nnumide,
        p.falta,
        TO_DATE(p.fnacimi, 'YYYY-MM-DD') AS fecha_nacimiento,
        CAST(p.fmovimi AS DATE) AS fmovimi_persona,
        ROW_NUMBER() OVER (
            PARTITION BY p.sperson
            ORDER BY
                (
                    CASE WHEN NULLIF(TRIM(CAST(p.nnumide AS VARCHAR(50))), '') IS NOT NULL THEN 1 ELSE 0 END +
                    CASE WHEN p.ctipide IS NOT NULL THEN 1 ELSE 0 END +
                    CASE WHEN p.falta IS NOT NULL THEN 1 ELSE 0 END +
                    CASE WHEN p.fnacimi IS NOT NULL THEN 1 ELSE 0 END
                ) DESC,
                p.fmovimi DESC
        ) AS rn
    FROM gde_adp_ods.axis_per_personas p
),
per_personas_filtrado AS (
    SELECT
        sperson,
        ctipide,
        nnumide,
        falta,
        fecha_nacimiento,
        fmovimi_persona
    FROM per_personas_base
    WHERE rn = 1
),
detper_base AS (
    SELECT
        d.sperson,
        TRIM(
            COALESCE(NULLIF(TRIM(d.tnombre1), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(d.tnombre2), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(d.tapelli1), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(d.tapelli2), ''), '')
        ) AS nombre_completo,
        TRY_CAST(d.ingresos AS NUMERIC(18,2)) AS ingresos,
        TRY_CAST(d.egresos AS NUMERIC(18,2)) AS egresos,
        CAST(d.fmovimi AS DATE) AS fmovimi_financiera,
        ROW_NUMBER() OVER (
            PARTITION BY d.sperson
            ORDER BY d.fmovimi DESC
        ) AS rn
    FROM gde_adp_ods.axis_per_detper d
),
detper_filtrado AS (
    SELECT
        sperson,
        nombre_completo,
        ingresos,
        egresos,
        fmovimi_financiera
    FROM detper_base
    WHERE rn = 1
),
correo_base AS (
    SELECT
        c.sperson,
        TRIM(CAST(c.tvalcon AS VARCHAR(200))) AS correo,
        CAST(c.fmovimi AS DATE) AS fmovimi_email,
        ROW_NUMBER() OVER (
            PARTITION BY c.sperson
            ORDER BY c.fmovimi DESC
        ) AS rn
    FROM gde_adp_ods.axis_per_contactos c
    WHERE c.ctipcon = 3
      AND NULLIF(TRIM(CAST(c.tvalcon AS VARCHAR(200))), '') IS NOT NULL
      AND POSITION('@' IN CAST(c.tvalcon AS VARCHAR(200))) > 1
),
correo_filtrado AS (
    SELECT
        sperson,
        correo,
        fmovimi_email
    FROM correo_base
    WHERE rn = 1
),
celular_base AS (
    SELECT
        c.sperson,
        TRIM(CAST(c.tvalcon AS VARCHAR(30))) AS celular,
        CAST(c.fmovimi AS DATE) AS fmovimi_celular,
        ROW_NUMBER() OVER (
            PARTITION BY c.sperson
            ORDER BY c.fmovimi DESC
        ) AS rn
    FROM gde_adp_ods.axis_per_contactos c
    WHERE c.ctipcon IN (5, 6)
      AND NULLIF(TRIM(CAST(c.tvalcon AS VARCHAR(30))), '') IS NOT NULL
      AND LENGTH(TRIM(CAST(c.tvalcon AS VARCHAR(30)))) >= 7
),
celular_filtrado AS (
    SELECT
        sperson,
        celular,
        fmovimi_celular
    FROM celular_base
    WHERE rn = 1
),
per_contactos_filtrado AS (
    SELECT
        COALESCE(cr.sperson, ce.sperson) AS sperson,
        cr.correo,
        cr.fmovimi_email,
        ce.celular,
        ce.fmovimi_celular
    FROM correo_filtrado cr
    FULL OUTER JOIN celular_filtrado ce
        ON cr.sperson = ce.sperson
),
as400_base AS (
    SELECT
        CASE
            WHEN a.tipo_identifi_clie = 'E' THEN 33
            WHEN a.tipo_identifi_clie = 'T' THEN 34
            WHEN a.tipo_identifi_clie = 'R' THEN 35
            WHEN a.tipo_identifi_clie = 'C' THEN 36
            WHEN a.tipo_identifi_clie = 'N' THEN 37
            WHEN a.tipo_identifi_clie = 'U' THEN 38
            WHEN a.tipo_identifi_clie = 'P' THEN 40
            WHEN a.tipo_identifi_clie = 'D' THEN 44
            ELSE 0
        END AS tipo_id,
        CAST(a.nro_identifi_clie AS VARCHAR(50)) AS nro_identifi_clie,
        TRIM(
            COALESCE(NULLIF(TRIM(a.nombre_1), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(a.nombre_2), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(a.apellido_1), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(a.apellido_2), ''), '')
        ) AS nombre_completo,
        CAST(a.telefono AS VARCHAR(30)) AS telefono,
        CAST(a.email AS VARCHAR(200)) AS email,
        TO_DATE(a.fecha_nacimiento, 'YYYY-MM-DD') AS fecha_nacimiento,
        CAST(a.fecha_ejecucion_dwh AS DATE) AS fecha_ejecucion_dwh,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(a.nro_identifi_clie AS VARCHAR(50))
            ORDER BY a.fecha_ejecucion_dwh DESC
        ) AS rn
    FROM gde_adp_ods.as400_dwh_clientes a
),
as400_filtrado AS (
    SELECT
        tipo_id,
        nro_identifi_clie,
        nombre_completo,
        telefono,
        email,
        fecha_nacimiento,
        fecha_ejecucion_dwh
    FROM as400_base
    WHERE rn = 1
),
sarlaft_base AS (
    SELECT
        CAST(s.num_identificacion AS VARCHAR(50)) AS num_identificacion,
        s.par_tipo_id,
        TRIM(
            COALESCE(NULLIF(TRIM(s.nombre1_basico), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(s.nombre2_basico), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(s.apellido1_basico), ''), '') || ' ' ||
            COALESCE(NULLIF(TRIM(s.apellido2_basico), ''), '')
        ) AS nombre_completo,
        TO_DATE(s.fecha_nacimiento, 'YYYY-MM-DD') AS fecha_nacimiento,
        CAST(s.email AS VARCHAR(200)) AS email,
        CAST(s.celular AS VARCHAR(30)) AS celular,
        TRY_CAST(s.total_activos AS NUMERIC(18,2)) AS total_activos,
        TRY_CAST(s.total_pasivos AS NUMERIC(18,2)) AS total_pasivos,
        TRY_CAST(s.ingreso_laboral AS NUMERIC(18,2)) AS ingreso_laboral,
        TRY_CAST(s.gasto_financiero AS NUMERIC(18,2)) AS total_egresos,
        CAST(s.fecha_diligenciamiento AS DATE) AS fecha_diligenciamiento,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(s.num_identificacion AS VARCHAR(50))
            ORDER BY s.fecha_diligenciamiento DESC
        ) AS rn
    FROM dp_dm_compliance.vm_formulario_sarlaft s
),
sarlaft_filtrado AS (
    SELECT
        num_identificacion,
        par_tipo_id,
        nombre_completo,
        fecha_nacimiento,
        email,
        celular,
        total_activos,
        total_pasivos,
        ingreso_laboral,
        total_egresos,
        fecha_diligenciamiento
    FROM sarlaft_base
    WHERE rn = 1
),
compra_datos_base AS (
    SELECT
        CAST(sct.numero_id AS VARCHAR(50)) AS numero_id,
        sct.tipo_id,
        sct.nombre_completo,
        CAST(sct.email AS VARCHAR(200)) AS email,
        CAST(sct.celular AS VARCHAR(30)) AS celular,
        TRY_CAST(sct.total_activos_sct AS NUMERIC(18,2)) AS total_activos_sct,
        TRY_CAST(sct.total_pasivos_sct AS NUMERIC(18,2)) AS total_pasivos_sct,
        TRY_CAST(sct.total_ingresos_sct AS NUMERIC(18,2)) AS total_ingresos_sct,
        TRY_CAST(sct.total_egresos_scs AS NUMERIC(18,2)) AS total_egresos_scs,
        CAST(sct.fecha_modificacion AS DATE) AS fecha_modificacion,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(sct.numero_id AS VARCHAR(50))
            ORDER BY sct.fecha_modificacion DESC
        ) AS rn
    FROM co_sandbox_datos.vw_compra_sct_unificada sct
),
compra_datos_filtrado AS (
    SELECT
        numero_id,
        tipo_id,
        nombre_completo,
        email,
        celular,
        total_activos_sct,
        total_pasivos_sct,
        total_ingresos_sct,
        total_egresos_scs,
        fecha_modificacion
    FROM compra_datos_base
    WHERE rn = 1
),
pep_base AS (
    SELECT
        CAST(pep.identificationnumber AS VARCHAR(50)) AS identificationnumber,
        pep."politicamente expuesto" AS politicamente_expuesto,
        CAST(pep."fecha de actualizacion" AS DATE) AS fecha_actualizacion_pep,
        TRY_CAST(
            NULLIF(
                REGEXP_REPLACE(CAST(pep.ingresos AS VARCHAR(100)), '[^0-9.-]', ''),
                ''
            ) AS NUMERIC(18,2)
        ) AS ingresos_pirani,
        TRY_CAST(
            NULLIF(
                REGEXP_REPLACE(CAST(pep.egresos AS VARCHAR(100)), '[^0-9.-]', ''),
                ''
            ) AS NUMERIC(18,2)
        ) AS egresos_pirani,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(pep.identificationnumber AS VARCHAR(50))
            ORDER BY CAST(pep."fecha de actualizacion" AS DATE) DESC
        ) AS rn
    FROM co_sandbox_datos.{TABLA_PEP} pep
),
pep_filtrado AS (
    SELECT
        identificationnumber,
        politicamente_expuesto,
        fecha_actualizacion_pep,
        ingresos_pirani,
        egresos_pirani
    FROM pep_base
    WHERE rn = 1
),
reporte_base AS (
    SELECT
        CASE
            WHEN COALESCE(sct.tipo_id, s.par_tipo_id, p.ctipide, a.tipo_id) IN (24, 33, 34, 35, 36, 38, 40, 44, 46, 48) THEN 1
            WHEN COALESCE(sct.tipo_id, s.par_tipo_id, p.ctipide, a.tipo_id) = 37 THEN 2
            ELSE NULL
        END AS tipo_persona,
        v.policy_holder,
        COALESCE(sct.tipo_id, s.par_tipo_id, p.ctipide, a.tipo_id) AS tipo_id,
        COALESCE(sct.numero_id, s.num_identificacion, p.nnumide, a.nro_identifi_clie) AS numero_documento,
        COALESCE(
            NULLIF(TRIM(sct.nombre_completo), ''),
            NULLIF(TRIM(s.nombre_completo), ''),
            NULLIF(TRIM(d.nombre_completo), ''),
            NULLIF(TRIM(a.nombre_completo), '')
        ) AS nombres_completos,
        COALESCE(s.fecha_nacimiento, p.fecha_nacimiento, a.fecha_nacimiento) AS fecha_nacimiento,
        p.falta AS fecha_vinculacion,
        COALESCE(
            NULLIF(TRIM(sct.celular), ''),
            NULLIF(TRIM(s.celular), ''),
            NULLIF(TRIM(c.celular), ''),
            NULLIF(TRIM(a.telefono), '')
        ) AS celular,
        COALESCE(
            NULLIF(TRIM(sct.email), ''),
            NULLIF(TRIM(s.email), ''),
            NULLIF(TRIM(c.correo), ''),
            NULLIF(TRIM(a.email), '')
        ) AS correo_electronico,
        COALESCE(sct.total_activos_sct, s.total_activos) AS total_activos,
        COALESCE(sct.total_pasivos_sct, s.total_pasivos) AS total_pasivos,
        COALESCE(sct.total_ingresos_sct, s.ingreso_laboral, d.ingresos) AS total_ingresos,
        COALESCE(sct.total_egresos_scs, s.total_egresos, d.egresos) AS total_egresos,
        CASE
            WHEN s.num_identificacion IS NOT NULL THEN 1
            ELSE 0
        END AS declara_sarlaft,
        CASE
            WHEN pep.identificationnumber IS NOT NULL THEN 'Intensificado'
            WHEN s.fecha_diligenciamiento IS NOT NULL THEN 'Obligado'
            ELSE 'Ordinario'
        END AS clasificacion,
        CAST(
            CASE
                WHEN pep.identificationnumber IS NOT NULL THEN 365
                WHEN s.fecha_diligenciamiento IS NOT NULL THEN 1095
                ELSE 1095
            END AS NUMERIC(10,0)
        ) AS perfil_riesgo,
        pep.politicamente_expuesto AS clientes_pep,
        pep.ingresos_pirani,
pep.egresos_pirani,
        /* Fechas de actualización por atributo */
        CASE
            WHEN sct.numero_id IS NOT NULL THEN sct.fecha_modificacion
            WHEN s.num_identificacion IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN p.nnumide IS NOT NULL THEN p.fmovimi_persona
            WHEN a.nro_identifi_clie IS NOT NULL THEN a.fecha_ejecucion_dwh
            ELSE NULL
        END AS fecha_actualizacion_numero_documento,
        CASE
            WHEN NULLIF(TRIM(sct.nombre_completo), '') IS NOT NULL THEN sct.fecha_modificacion
            WHEN NULLIF(TRIM(s.nombre_completo), '') IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN NULLIF(TRIM(d.nombre_completo), '') IS NOT NULL THEN d.fmovimi_financiera
            WHEN NULLIF(TRIM(a.nombre_completo), '') IS NOT NULL THEN a.fecha_ejecucion_dwh
            ELSE NULL
        END AS fecha_actualizacion_nombres,
        CASE
            WHEN s.fecha_nacimiento IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN p.fecha_nacimiento IS NOT NULL THEN p.fmovimi_persona
            WHEN a.fecha_nacimiento IS NOT NULL THEN a.fecha_ejecucion_dwh
            ELSE NULL
        END AS fecha_actualizacion_fecha_nacimiento,
        p.fmovimi_persona AS fecha_actualizacion_fecha_vinculacion,
        CASE
            WHEN NULLIF(TRIM(sct.celular), '') IS NOT NULL THEN sct.fecha_modificacion
            WHEN NULLIF(TRIM(s.celular), '') IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN NULLIF(TRIM(c.celular), '') IS NOT NULL THEN c.fmovimi_celular
            WHEN NULLIF(TRIM(a.telefono), '') IS NOT NULL THEN a.fecha_ejecucion_dwh
            ELSE NULL
        END AS fecha_actualizacion_celular,
        CASE
            WHEN NULLIF(TRIM(sct.email), '') IS NOT NULL THEN sct.fecha_modificacion
            WHEN NULLIF(TRIM(s.email), '') IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN NULLIF(TRIM(c.correo), '') IS NOT NULL THEN c.fmovimi_email
            WHEN NULLIF(TRIM(a.email), '') IS NOT NULL THEN a.fecha_ejecucion_dwh
            ELSE NULL
        END AS fecha_actualizacion_correo,
        CASE
            WHEN sct.total_activos_sct IS NOT NULL THEN sct.fecha_modificacion
            WHEN s.total_activos IS NOT NULL THEN s.fecha_diligenciamiento
            ELSE p.fmovimi_persona
        END AS fecha_actualizacion_activos,
        CASE
            WHEN sct.total_pasivos_sct IS NOT NULL THEN sct.fecha_modificacion
            WHEN s.total_pasivos IS NOT NULL THEN s.fecha_diligenciamiento
            ELSE p.fmovimi_persona
        END AS fecha_actualizacion_pasivos,
        CASE
            WHEN sct.total_ingresos_sct IS NOT NULL THEN sct.fecha_modificacion
            WHEN s.ingreso_laboral IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN d.ingresos IS NOT NULL THEN d.fmovimi_financiera
            ELSE p.fmovimi_persona
        END AS fecha_actualizacion_ingresos,
        CASE
            WHEN sct.total_egresos_scs IS NOT NULL THEN sct.fecha_modificacion
            WHEN s.total_egresos IS NOT NULL THEN s.fecha_diligenciamiento
            WHEN d.egresos IS NOT NULL THEN d.fmovimi_financiera
            ELSE p.fmovimi_persona
        END AS fecha_actualizacion_egresos
    FROM vigentes_filtrado v
    LEFT JOIN per_personas_filtrado p
        ON v.policy_holder = p.sperson
    LEFT JOIN detper_filtrado d
        ON p.sperson = d.sperson
    LEFT JOIN per_contactos_filtrado c
        ON p.sperson = c.sperson
    LEFT JOIN sarlaft_filtrado s
        ON p.nnumide = s.num_identificacion
    LEFT JOIN compra_datos_filtrado sct
        ON p.nnumide = sct.numero_id
    LEFT JOIN as400_filtrado a
        ON p.nnumide = a.nro_identifi_clie
    LEFT JOIN pep_filtrado pep
        ON COALESCE(sct.numero_id, s.num_identificacion, p.nnumide, a.nro_identifi_clie) = pep.identificationnumber
)
SELECT *
FROM reporte_base;
'''

run_statement('DROP TABLE IF EXISTS tmp_reporte_base;')
run_statement(BLOQUE_A)
print('tmp_reporte_base creada.')

tmp_reporte_base creada.


In [7]:
# Chequeo rápido del bloque A (equivale a las "PRUEBAS PARA BLOQUE A" del SQL original).
run_query(f'''
select
    (select count(*) from tmp_reporte_base)                                            as clientes_reporte_base,
    (select count(distinct policy_holder)
       from co_sandbox_datos.fact_vigentes_total_historico
      where accountable_period = '{PERIODO}')                                          as tomadores_vigentes,
    (select count(*) from tmp_reporte_base where tipo_persona = 1)                     as persona_natural,
    (select count(*) from tmp_reporte_base where tipo_persona = 2)                     as persona_juridica,
    (select count(*) from tmp_reporte_base where tipo_persona is null)                 as sin_clasificar
''')

,clientes_reporte_base,tomadores_vigentes,persona_natural,persona_juridica,sin_clasificar
0,257261,257260,228603,28623,35


## Bloque B — Reglas de calidad por cliente (`tmp_reglas_reporte_base`)

Agrega a cada cliente las banderas `rg_completitud_*`, `rg_validez_*`, `rg_unicidad_*`,
`rg_precision_*`, `rg_consistencia_*` y `rg_oportunidad_*` (1 = cumple, 0 = no cumple;
`rg_consistencia_*` deja **NULL** cuando Pirani no tiene valor con qué comparar).

In [8]:
# Bloque B tal cual el SQL original (raw string: los regex usan {10} y {2,}).
BLOQUE_B = r'''
CREATE TEMP TABLE tmp_reglas_reporte_base AS
WITH rb_norm AS (
    SELECT
        rb.*,

        /* =========================
           LLAVES NORMALIZADAS PARA UNICIDAD
           ========================= */

        CASE
            WHEN rb.numero_documento IS NOT NULL
             AND BTRIM(rb.numero_documento) <> ''
            THEN BTRIM(rb.numero_documento)
            ELSE NULL
        END AS key_numero_documento,

        CASE
            WHEN rb.nombres_completos IS NOT NULL
             AND BTRIM(rb.nombres_completos) <> ''
            THEN REGEXP_REPLACE(
                    BTRIM(UPPER(rb.nombres_completos)),
                    '[ ]+',
                    ' '
                 )
            ELSE NULL
        END AS key_nombres_completos,

        CASE
            WHEN rb.celular IS NOT NULL
             AND BTRIM(rb.celular) <> ''
             AND REGEXP_REPLACE(rb.celular, '[^0-9]', '') <> ''
            THEN REGEXP_REPLACE(rb.celular, '[^0-9]', '')
            ELSE NULL
        END AS key_celular,

        CASE
            WHEN rb.correo_electronico IS NOT NULL
             AND BTRIM(rb.correo_electronico) <> ''
            THEN LOWER(BTRIM(rb.correo_electronico))
            ELSE NULL
        END AS key_correo_electronico

    FROM tmp_reporte_base rb
)
SELECT
    rb.*,

    /* =========================
       COMPLETITUD
       ========================= */
    CASE WHEN rb.numero_documento IS NOT NULL AND BTRIM(rb.numero_documento) <> '' THEN 1 ELSE 0 END AS rg_completitud_numero_documento,
    CASE WHEN rb.nombres_completos IS NOT NULL AND BTRIM(rb.nombres_completos) <> '' THEN 1 ELSE 0 END AS rg_completitud_nombres,
    CASE WHEN rb.fecha_nacimiento IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_fecha_nacimiento,
    CASE WHEN rb.fecha_vinculacion IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_fecha_vinculacion,
    CASE WHEN rb.celular IS NOT NULL AND BTRIM(rb.celular) <> '' THEN 1 ELSE 0 END AS rg_completitud_celular,
    CASE WHEN rb.correo_electronico IS NOT NULL AND BTRIM(rb.correo_electronico) <> '' THEN 1 ELSE 0 END AS rg_completitud_correo,
    CASE WHEN rb.total_activos IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_activos,
    CASE WHEN rb.total_pasivos IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_pasivos,
    CASE WHEN rb.total_ingresos IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_ingresos,
    CASE WHEN rb.total_egresos IS NOT NULL THEN 1 ELSE 0 END AS rg_completitud_egresos,

    /* =========================
       VALIDEZ
       ========================= */
    CASE
        WHEN rb.numero_documento IS NOT NULL
         AND BTRIM(rb.numero_documento) <> ''
         AND POSITION(' ' IN rb.numero_documento) = 0
        THEN 1 ELSE 0
    END AS rg_validez_numero_documento,

    CASE
        WHEN rb.nombres_completos IS NOT NULL
         AND BTRIM(rb.nombres_completos) <> ''
         AND REGEXP_INSTR(rb.nombres_completos, '^[A-Za-zÁÉÍÓÚáéíóúÑñ ]+$') = 1
        THEN 1 ELSE 0
    END AS rg_validez_nombres,

    CASE
        WHEN rb.fecha_nacimiento IS NULL THEN 0
        WHEN rb.fecha_nacimiento > CURRENT_DATE THEN 0
        WHEN rb.fecha_nacimiento < DATE '1940-01-01' THEN 0
        ELSE 1
    END AS rg_validez_fecha_nacimiento,

    CASE
        WHEN rb.fecha_vinculacion IS NULL THEN 0
        WHEN rb.fecha_vinculacion > CURRENT_DATE THEN 0
        WHEN rb.fecha_nacimiento IS NOT NULL AND rb.fecha_vinculacion < rb.fecha_nacimiento THEN 0
        ELSE 1
    END AS rg_validez_fecha_vinculacion,

    CASE
        WHEN rb.celular IS NOT NULL
         AND REGEXP_INSTR(REGEXP_REPLACE(rb.celular, '[^0-9]', ''), '^[0-9]{10}$') = 1
        THEN 1 ELSE 0
    END AS rg_validez_celular,

    CASE
        WHEN rb.correo_electronico IS NOT NULL
         AND REGEXP_INSTR(
                LOWER(BTRIM(rb.correo_electronico)),
                '^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$'
             ) = 1
        THEN 1 ELSE 0
    END AS rg_validez_correo,

    CASE WHEN rb.total_activos IS NOT NULL AND rb.total_activos >= 0 THEN 1 ELSE 0 END AS rg_validez_activos,
    CASE WHEN rb.total_pasivos IS NOT NULL AND rb.total_pasivos >= 0 THEN 1 ELSE 0 END AS rg_validez_pasivos,
    CASE WHEN rb.total_ingresos IS NOT NULL AND rb.total_ingresos >= 0 THEN 1 ELSE 0 END AS rg_validez_ingresos,
    CASE WHEN rb.total_egresos IS NOT NULL AND rb.total_egresos >= 0 THEN 1 ELSE 0 END AS rg_validez_egresos,

    /* =========================
       UNICIDAD
       ========================= */

    CASE
        WHEN rb.key_numero_documento IS NOT NULL
         AND COUNT(*) OVER (
                PARTITION BY rb.tipo_id, rb.key_numero_documento
             ) = 1
        THEN 1 ELSE 0
    END AS rg_unicidad_numero_documento,

    CASE
        WHEN rb.key_nombres_completos IS NOT NULL
         AND COUNT(*) OVER (
                PARTITION BY rb.key_nombres_completos
             ) = 1
        THEN 1 ELSE 0
    END AS rg_unicidad_nombres,

    CASE
        WHEN rb.key_celular IS NOT NULL
         AND COUNT(*) OVER (
                PARTITION BY rb.key_celular
             ) = 1
        THEN 1 ELSE 0
    END AS rg_unicidad_celular,

    CASE
        WHEN rb.key_correo_electronico IS NOT NULL
         AND COUNT(*) OVER (
                PARTITION BY rb.key_correo_electronico
             ) = 1
        THEN 1 ELSE 0
    END AS rg_unicidad_correo,

    /* =========================
       PRECISION
       ========================= */
    CASE
        WHEN rb.total_ingresos IS NOT NULL
         AND rb.total_ingresos > 0
         AND rb.total_egresos IS NOT NULL
         AND rb.total_ingresos >= rb.total_egresos
        THEN 1 ELSE 0
    END AS rg_precision_ingresos,

    CASE
        WHEN rb.total_egresos IS NOT NULL
         AND rb.total_egresos > 0
         AND rb.total_ingresos IS NOT NULL
         AND rb.total_ingresos >= rb.total_egresos
        THEN 1 ELSE 0
    END AS rg_precision_egresos,

/* =========================
       CONSISTENCIA
       Si la fuente oficial no tiene valor, se deja NULL porque no hay base
       para comparar consistencia. Si la fuente oficial tiene valor y tu
       reporte viene NULL, se marca 0.
       ========================= */

    CASE
        WHEN rb.ingresos_pirani IS NULL THEN NULL
        WHEN rb.total_ingresos IS NULL THEN 0
        WHEN ABS(rb.total_ingresos - rb.ingresos_pirani) <= 1 THEN 1
        ELSE 0
    END AS rg_consistencia_ingresos,

    CASE
        WHEN rb.egresos_pirani IS NULL THEN NULL
        WHEN rb.total_egresos IS NULL THEN 0
        WHEN ABS(rb.total_egresos - rb.egresos_pirani) <= 1 THEN 1
        ELSE 0
    END AS rg_consistencia_egresos,

    /* =========================
       OPORTUNIDAD
       ========================= */
    CASE
        WHEN rb.fecha_actualizacion_numero_documento IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_numero_documento, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_numero_documento,

    CASE
        WHEN rb.fecha_actualizacion_nombres IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_nombres, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_nombres,

    CASE
        WHEN rb.fecha_actualizacion_fecha_nacimiento IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_fecha_nacimiento, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_fecha_nacimiento,

    CASE
        WHEN rb.fecha_actualizacion_fecha_vinculacion IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_fecha_vinculacion, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_fecha_vinculacion,

    CASE
        WHEN rb.fecha_actualizacion_celular IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_celular, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_celular,

    CASE
        WHEN rb.fecha_actualizacion_correo IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_correo, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_correo,

    CASE
        WHEN rb.fecha_actualizacion_activos IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_activos, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_activos,

    CASE
        WHEN rb.fecha_actualizacion_pasivos IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_pasivos, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_pasivos,

    CASE
        WHEN rb.fecha_actualizacion_ingresos IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_ingresos, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_ingresos,

    CASE
        WHEN rb.fecha_actualizacion_egresos IS NOT NULL
         AND (DATEDIFF(day, rb.fecha_actualizacion_egresos, CURRENT_DATE) + 30) <= rb.perfil_riesgo
        THEN 1 ELSE 0
    END AS rg_oportunidad_egresos

FROM rb_norm rb;
'''

run_statement('DROP TABLE IF EXISTS tmp_reglas_reporte_base;')
run_statement(BLOQUE_B)
print('tmp_reglas_reporte_base creada.')

tmp_reglas_reporte_base creada.


## Bloque C — Métricas por atributo × dimensión × tipo de persona

Del bloque C **solo se traen agregados** (porcentajes); ningún dato personal sale de la base.
Aquí es donde `pct_consistencia` aparece en `NULL` para persona jurídica → ver el diagnóstico.

In [9]:
BLOQUE_C = r'''
WITH metricas AS (
    /* =========================
       COMPLETITUD
       ========================= */
    SELECT
        tipo_persona,
        'numero_documento' AS atributo,
        'completitud' AS dimension,
        AVG(rg_completitud_numero_documento::DECIMAL(18,6)) * 100 AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'completitud',
        AVG(rg_completitud_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_nacimiento',
        'completitud',
        AVG(rg_completitud_fecha_nacimiento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_vinculacion',
        'completitud',
        AVG(rg_completitud_fecha_vinculacion::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'completitud',
        AVG(rg_completitud_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'completitud',
        AVG(rg_completitud_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'completitud',
        AVG(rg_completitud_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'completitud',
        AVG(rg_completitud_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'completitud',
        AVG(rg_completitud_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'completitud',
        AVG(rg_completitud_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    /* =========================
       VALIDEZ
       ========================= */
    UNION ALL
    SELECT
        tipo_persona,
        'numero_documento',
        'validez',
        AVG(rg_validez_numero_documento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'validez',
        AVG(rg_validez_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_nacimiento',
        'validez',
        AVG(rg_validez_fecha_nacimiento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_vinculacion',
        'validez',
        AVG(rg_validez_fecha_vinculacion::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'validez',
        AVG(rg_validez_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'validez',
        AVG(rg_validez_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'validez',
        AVG(rg_validez_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'validez',
        AVG(rg_validez_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'validez',
        AVG(rg_validez_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'validez',
        AVG(rg_validez_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    /* =========================
       UNICIDAD
       ========================= */    
UNION ALL
    SELECT
        tipo_persona,
        'numero_documento',
        'unicidad',
        AVG(rg_unicidad_numero_documento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'unicidad',
        AVG(rg_unicidad_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'unicidad',
        AVG(rg_unicidad_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'unicidad',
        AVG(rg_unicidad_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    /* =========================
       PRECISION
       ========================= */
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'precision',
        AVG(rg_precision_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'precision',
        AVG(rg_precision_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    /* =========================
       CONSISTENCIA
       ========================= */

    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos' AS atributo,
        'consistencia' AS dimension,
        LEAST(
            100,
            GREATEST(
                0,
                (
                    1 - (
                        ABS(
                            SUM(COALESCE(total_ingresos, 0)) 
                            - SUM(COALESCE(ingresos_pirani, 0))
                        )
                        / NULLIF(SUM(COALESCE(ingresos_pirani, 0)), 0)
                    )
                ) * 100
            )
        ) AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
      AND ingresos_pirani IS NOT NULL
    GROUP BY tipo_persona

    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos' AS atributo,
        'consistencia' AS dimension,
        LEAST(
            100,
            GREATEST(
                0,
                (
                    1 - (
                        ABS(
                            SUM(COALESCE(total_egresos, 0)) 
                            - SUM(COALESCE(egresos_pirani, 0))
                        )
                        / NULLIF(SUM(COALESCE(egresos_pirani, 0)), 0)
                    )
                ) * 100
            )
        ) AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
      AND egresos_pirani IS NOT NULL
    GROUP BY tipo_persona
    /* =========================
       OPORTUNIDAD
       ========================= */
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'oportunidad',
        AVG(rg_oportunidad_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'oportunidad',
        AVG(rg_oportunidad_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'oportunidad',
        AVG(rg_oportunidad_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'oportunidad',
        AVG(rg_oportunidad_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'oportunidad',
        AVG(rg_oportunidad_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'oportunidad',
        AVG(rg_oportunidad_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
)
SELECT
    tipo_persona,
    CASE
        WHEN tipo_persona = 1 THEN 'Persona natural'
        WHEN tipo_persona = 2 THEN 'Persona juridica'
        ELSE 'No clasificada'
    END AS desc_tipo_persona,
    atributo,
    ROUND(MAX(CASE WHEN dimension = 'completitud' THEN porcentaje END), 2) AS pct_completitud,
    ROUND(MAX(CASE WHEN dimension = 'validez' THEN porcentaje END), 2) AS pct_validez,
    ROUND(MAX(CASE WHEN dimension = 'unicidad' THEN porcentaje END), 2) AS pct_unicidad,
    ROUND(MAX(CASE WHEN dimension = 'precision' THEN porcentaje END), 2) AS pct_precision,
    ROUND(MAX(CASE WHEN dimension = 'consistencia' THEN porcentaje END), 2) AS pct_consistencia,
    ROUND(MAX(CASE WHEN dimension = 'oportunidad' THEN porcentaje END), 2) AS pct_oportunidad,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'completitud' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_completitud_90,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'validez' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_validez_90,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'unicidad' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_unicidad_90,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'precision' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_precision_90,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'consistencia' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_consistencia_90,
    CASE WHEN ROUND(MAX(CASE WHEN dimension = 'oportunidad' THEN porcentaje END), 2) >= 90 THEN 1 ELSE 0 END AS cumple_oportunidad_90
FROM metricas
GROUP BY tipo_persona, atributo
ORDER BY tipo_persona, atributo;
'''

df_metricas = run_query(BLOQUE_C)
df_metricas

,tipo_persona,desc_tipo_persona,atributo,pct_completitud,pct_validez,pct_unicidad,pct_precision,pct_consistencia,pct_oportunidad,cumple_completitud_90,cumple_validez_90,cumple_unicidad_90,cumple_precision_90,cumple_consistencia_90,cumple_oportunidad_90
0,1,Persona natural,celular,98.91,98.02,94.06,None,None,30.90,1,1,1,0,0,0
1,1,Persona natural,correo_electronico,98.36,98.24,91.16,None,None,31.18,1,1,1,0,0,0
2,1,Persona natural,fecha_nacimiento,94.79,91.84,None,None,None,None,1,1,0,0,0,0
3,1,Persona natural,fecha_vinculacion,100.00,99.99,None,None,None,None,1,1,0,0,0,0
4,1,Persona natural,nombres_completos,100.00,98.53,99.54,None,None,None,1,1,1,0,0,0
5,1,Persona natural,numero_documento,100.00,99.75,99.91,None,None,None,1,1,1,0,0,0
6,1,Persona natural,total_activos,31.50,31.50,None,None,None,10.28,0,0,0,0,0,0
7,1,Persona natural,total_egresos,82.61,82.61,None,64.03,99.47,12.02,0,0,0,0,1,0
8,1,Persona natural,total_ingresos,88.85,88.85,None,70.29,98.29,6.93,0,0,0,0,1,0
9,1,Persona natural,total_pasivos,36.72,36.72,None,None,None,9.89,0,0,0,0,0,0


In [10]:
# Vista enfocada: cómo queda la dimensión consistencia por tipo de persona.
df_metricas[df_metricas['atributo'].isin(['total_ingresos', 'total_egresos'])][
    ['desc_tipo_persona', 'atributo', 'pct_completitud', 'pct_precision', 'pct_consistencia']
]

,desc_tipo_persona,atributo,pct_completitud,pct_precision,pct_consistencia
7,Persona natural,total_egresos,82.61,64.03,99.47
8,Persona natural,total_ingresos,88.85,70.29,98.29
17,Persona juridica,total_egresos,97.14,87.81,0.00
18,Persona juridica,total_ingresos,97.71,91.62,0.00


## Bloque SFC — Salida en formato de la Superintendencia Financiera

Mismos indicadores, codificados (`tipo_persona_cod`, `campo_cod`, `dimension_cod`, `valor`).

> Diferencia con el SQL original: allí es `CREATE TABLE` (tabla **permanente** en el esquema por
> defecto, que además hace fallar la segunda corrida porque la tabla ya existe). Aquí se ejecuta
> como `CREATE TEMP TABLE` — hallazgo #2 al final.

In [11]:
BLOQUE_SFC = r'''
DROP TABLE IF EXISTS tmp_reporte_sfc_calidad_datos;
CREATE TEMP TABLE tmp_reporte_sfc_calidad_datos AS
WITH metricas AS (
    /* =========================
       COMPLETITUD
       ========================= */
    SELECT
        tipo_persona,
        'numero_documento' AS atributo,
        'completitud' AS dimension,
        AVG(rg_completitud_numero_documento::DECIMAL(18,6)) * 100 AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'completitud',
        AVG(rg_completitud_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_nacimiento',
        'completitud',
        AVG(rg_completitud_fecha_nacimiento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_vinculacion',
        'completitud',
        AVG(rg_completitud_fecha_vinculacion::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'completitud',
        AVG(rg_completitud_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'completitud',
        AVG(rg_completitud_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'completitud',
        AVG(rg_completitud_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'completitud',
        AVG(rg_completitud_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'completitud',
        AVG(rg_completitud_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'completitud',
        AVG(rg_completitud_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    /* =========================
       VALIDEZ
       ========================= */
    UNION ALL
    SELECT
        tipo_persona,
        'numero_documento',
        'validez',
        AVG(rg_validez_numero_documento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'validez',
        AVG(rg_validez_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_nacimiento',
        'validez',
        AVG(rg_validez_fecha_nacimiento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_vinculacion',
        'validez',
        AVG(rg_validez_fecha_vinculacion::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'validez',
        AVG(rg_validez_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'validez',
        AVG(rg_validez_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'validez',
        AVG(rg_validez_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'validez',
        AVG(rg_validez_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'validez',
        AVG(rg_validez_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'validez',
        AVG(rg_validez_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    /* =========================
       UNICIDAD
       ========================= */    
UNION ALL
    SELECT
        tipo_persona,
        'numero_documento',
        'unicidad',
        AVG(rg_unicidad_numero_documento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'unicidad',
        AVG(rg_unicidad_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'unicidad',
        AVG(rg_unicidad_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'unicidad',
        AVG(rg_unicidad_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    /* =========================
       PRECISION
       ========================= */
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'precision',
        AVG(rg_precision_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'precision',
        AVG(rg_precision_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    
     /* =========================
       CONSISTENCIA
       ========================= */

    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos' AS atributo,
        'consistencia' AS dimension,
        LEAST(
            100,
            GREATEST(
                0,
                (
                    1 - (
                        ABS(
                            SUM(COALESCE(total_ingresos, 0)) 
                            - SUM(COALESCE(ingresos_pirani, 0))
                        )
                        / NULLIF(SUM(COALESCE(ingresos_pirani, 0)), 0)
                    )
                ) * 100
            )
        ) AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
      AND ingresos_pirani IS NOT NULL
    GROUP BY tipo_persona

    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos' AS atributo,
        'consistencia' AS dimension,
        LEAST(
            100,
            GREATEST(
                0,
                (
                    1 - (
                        ABS(
                            SUM(COALESCE(total_egresos, 0)) 
                            - SUM(COALESCE(egresos_pirani, 0))
                        )
                        / NULLIF(SUM(COALESCE(egresos_pirani, 0)), 0)
                    )
                ) * 100
            )
        ) AS porcentaje
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
      AND egresos_pirani IS NOT NULL
    GROUP BY tipo_persona   
    /* =========================
       OPORTUNIDAD
       ========================= */
    /*UNION ALL
    SELECT
        tipo_persona,
        'numero_documento',
        'oportunidad',
        AVG(rg_oportunidad_numero_documento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'nombres_completos',
        'oportunidad',
        AVG(rg_oportunidad_nombres::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_nacimiento',
        'oportunidad',
        AVG(rg_oportunidad_fecha_nacimiento::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'fecha_vinculacion',
        'oportunidad',
        AVG(rg_oportunidad_fecha_vinculacion::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona*/
    UNION ALL
    SELECT
        tipo_persona,
        'celular',
        'oportunidad',
        AVG(rg_oportunidad_celular::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'correo_electronico',
        'oportunidad',
        AVG(rg_oportunidad_correo::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_activos',
        'oportunidad',
        AVG(rg_oportunidad_activos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_pasivos',
        'oportunidad',
        AVG(rg_oportunidad_pasivos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_ingresos',
        'oportunidad',
        AVG(rg_oportunidad_ingresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
    UNION ALL
    SELECT
        tipo_persona,
        'total_egresos',
        'oportunidad',
        AVG(rg_oportunidad_egresos::DECIMAL(18,6)) * 100
    FROM tmp_reglas_reporte_base
    WHERE tipo_persona IN (1,2)
      AND declara_sarlaft = 1
    GROUP BY tipo_persona
)

SELECT
    tipo_persona AS tipo_persona_cod,

    CASE atributo
        WHEN 'numero_documento'     THEN 1
        WHEN 'nombres_completos'    THEN 2
        WHEN 'fecha_nacimiento'     THEN 3
        WHEN 'fecha_vinculacion'    THEN 4
        WHEN 'celular'              THEN 5
        WHEN 'correo_electronico'   THEN 6
        WHEN 'total_activos'        THEN 7
        WHEN 'total_pasivos'        THEN 8
        WHEN 'total_ingresos'       THEN 9
        WHEN 'total_egresos'        THEN 10
    END AS campo_cod,

    CASE dimension
        WHEN 'completitud'  THEN 1
        WHEN 'validez'      THEN 2
        WHEN 'unicidad'     THEN 3
        WHEN 'precision'    THEN 4
        WHEN 'consistencia' THEN 5
        WHEN 'oportunidad'  THEN 6
    END AS dimension_cod,

    ROUND(porcentaje, 2) AS valor

FROM metricas
ORDER BY
    tipo_persona_cod,
    campo_cod,
    dimension_cod;
'''

for sentencia in [s for s in BLOQUE_SFC.split(';\n') if s.strip()]:
    run_statement(sentencia)
df_sfc = run_query('select * from tmp_reporte_sfc_calidad_datos order by tipo_persona_cod, campo_cod, dimension_cod;')
df_sfc

,tipo_persona_cod,campo_cod,dimension_cod,valor
0,1,1,1,100.00
1,1,1,2,99.75
2,1,1,3,99.91
3,1,2,1,100.00
4,1,2,2,98.53
5,1,2,3,99.54
6,1,3,1,94.79
7,1,3,2,91.84
8,1,4,1,100.00
9,1,4,2,99.99


---

# Diagnóstico — la consistencia de persona jurídica

La métrica de consistencia del bloque C es:

```sql
SELECT tipo_persona, 'total_ingresos', 'consistencia',
       LEAST(100, GREATEST(0,
            (1 - ( ABS(SUM(COALESCE(total_ingresos,0)) - SUM(COALESCE(ingresos_pirani,0)))
                   / NULLIF(SUM(COALESCE(ingresos_pirani,0)), 0) )) * 100))
FROM tmp_reglas_reporte_base
WHERE tipo_persona IN (1,2)
  AND declara_sarlaft = 1
  AND ingresos_pirani IS NOT NULL      -- <== universo
GROUP BY tipo_persona
```

Hay **dos** caminos distintos hacia un resultado sospechoso, y hay que separarlos:

1. **`NULL`** — no existe ninguna fila con `tipo_persona = 2 AND declara_sarlaft = 1 AND
   ingresos_pirani IS NOT NULL`. Sin filas no hay grupo, el `UNION ALL` no aporta fila de
   `consistencia` para jurídica, y el `MAX(CASE WHEN dimension = 'consistencia' ...)` del pivot
   final devuelve `NULL`. **Ojo:** `pct_consistencia` también sale `NULL` —legítimamente— en los
   **otros 8 atributos**, porque esta dimensión solo se calcula para `total_ingresos` y
   `total_egresos`.
2. **`0.00`** — sí hay filas, pero la desviación relativa entre las dos sumas supera el 100 %
   (o `SUM(ingresos_pirani) = 0`, que vía `NULLIF` da `NULL` y `GREATEST(0, NULL)` lo aplana a 0
   en Redshift, ver D8). El `0.00` **no** significa "0 % de los clientes son consistentes":
   significa que la fórmula se salió de rango y se truncó.

Las celdas D1–D11 determinan en cuál de los dos casos está cada tipo de persona y por qué.

In [12]:
# D1 — Embudo de población por tipo de persona: dónde se queda la jurídica.
run_query('''
select
    coalesce(cast(tipo_persona as varchar), 'NULL (sin clasificar)')                        as tipo_persona,
    count(*)                                                                                as clientes,
    sum(case when declara_sarlaft = 1 then 1 else 0 end)                                    as con_form_sarlaft,
    sum(case when clasificacion = 'Intensificado' then 1 else 0 end)                        as cruzan_con_pirani,
    sum(case when ingresos_pirani is not null then 1 else 0 end)                            as con_ingresos_pirani,
    sum(case when egresos_pirani  is not null then 1 else 0 end)                            as con_egresos_pirani,
    sum(case when declara_sarlaft = 1 and ingresos_pirani is not null then 1 else 0 end)    as universo_consist_ingresos,
    sum(case when declara_sarlaft = 1 and egresos_pirani  is not null then 1 else 0 end)    as universo_consist_egresos
from tmp_reglas_reporte_base
group by 1
order by 1;
''')

,tipo_persona,clientes,con_form_sarlaft,cruzan_con_pirani,con_ingresos_pirani,con_egresos_pirani,universo_consist_ingresos,universo_consist_egresos
0,1,228603,98034,226953,226953,226953,97178,97178
1,2,28623,525,28456,28456,28456,522,522
2,NULL (sin clasificar),35,0,0,0,0,0,0


In [13]:
# D2 — La métrica exacta del bloque C, desarmada. Si la fila de tipo_persona = 2
#      NO aparece aquí, la causa es el camino (1): universo vacío.
run_query('''
select
    tipo_persona,
    count(*)                                        as filas_en_el_universo,
    sum(coalesce(total_ingresos, 0))                as suma_ingresos_reporte,
    sum(coalesce(ingresos_pirani, 0))               as suma_ingresos_pirani,
    sum(case when total_ingresos is null then 1 else 0 end) as ingresos_reporte_nulos,
    round(
        least(100, greatest(0,
            (1 - (abs(sum(coalesce(total_ingresos,0)) - sum(coalesce(ingresos_pirani,0)))
                  / nullif(sum(coalesce(ingresos_pirani,0)), 0))) * 100)), 2)  as pct_consistencia_calculado
from tmp_reglas_reporte_base
where tipo_persona in (1,2)
  and declara_sarlaft = 1
  and ingresos_pirani is not null
group by tipo_persona
order by tipo_persona;
''')

,tipo_persona,filas_en_el_universo,suma_ingresos_reporte,suma_ingresos_pirani,ingresos_reporte_nulos,pct_consistencia_calculado
0,1,97178,980711852116.00,964187921165.00,10829,98.29
1,2,522,2973929676533.00,230469534390.00,12,0.00


In [14]:
# D3 — ¿Los documentos de las personas jurídicas existen siquiera en la tabla Pirani?
#      (independiente del filtro declara_sarlaft)
run_query(f'''
select
    coalesce(cast(rb.tipo_persona as varchar), 'NULL') as tipo_persona,
    count(*)                                                              as clientes,
    sum(case when pep.identificationnumber is not null then 1 else 0 end) as cruzan_pirani,
    sum(case when pep.ingresos is not null then 1 else 0 end)             as pirani_con_ingresos_crudos,
    sum(case when pep.egresos  is not null then 1 else 0 end)             as pirani_con_egresos_crudos
from tmp_reporte_base rb
left join co_sandbox_datos.{TABLA_PEP} pep
    on rb.numero_documento = cast(pep.identificationnumber as varchar(50))
group by 1
order by 1;
''')

,tipo_persona,clientes,cruzan_pirani,pirani_con_ingresos_crudos,pirani_con_egresos_crudos
0,1,228671,227021,227021,227021
1,2,28653,28486,28486,28486
2,NULL,35,0,0,0


In [15]:
# D4 — Estructura de la tabla Pirani: ¿qué columnas trae y hay alguna de tipo de documento?
run_query(f'''
select column_name, data_type
from information_schema.columns
where table_schema = 'co_sandbox_datos'
  and table_name   = '{TABLA_PEP}'
order by ordinal_position;
''')

,column_name,data_type
0,fecha_cargue,timestamp without time zone
1,fecha_modificacion,timestamp without time zone
2,identificationtype,character varying
3,identificationnumber,character varying
4,tipo persona,character varying
5,nombre,character varying
6,primer apellido,character varying
7,segundo apellido,character varying
8,razon social,character varying
9,fecha de nacimiento,character varying


In [16]:
# D5 — ¿Se pierden valores al parsear ingresos/egresos de Pirani?
#      (el REGEXP_REPLACE quita todo lo que no sea 0-9 . -; un formato "1.234.567,89"
#       queda como "1.234.567.89" y el TRY_CAST devuelve NULL sin avisar)
run_query(f'''
select
    count(*)                                                                    as filas_pirani,
    sum(case when ingresos is null then 1 else 0 end)                           as ingresos_crudos_nulos,
    sum(case when ingresos is not null
              and try_cast(nullif(regexp_replace(cast(ingresos as varchar(100)), '[^0-9.-]', ''), '')
                           as numeric(18,2)) is null
             then 1 else 0 end)                                                 as ingresos_no_parseables,
    sum(case when egresos is null then 1 else 0 end)                            as egresos_crudos_nulos,
    sum(case when egresos is not null
              and try_cast(nullif(regexp_replace(cast(egresos as varchar(100)), '[^0-9.-]', ''), '')
                           as numeric(18,2)) is null
             then 1 else 0 end)                                                 as egresos_no_parseables
from co_sandbox_datos.{TABLA_PEP};
''')

,filas_pirani,ingresos_crudos_nulos,ingresos_no_parseables,egresos_crudos_nulos,egresos_no_parseables
0,357140,0,0,0,0


In [17]:
# D6 — Muestra de valores crudos de Pirani (para ver el formato real de ingresos/egresos).
run_query(f'''
select cast(ingresos as varchar(100)) as ingresos_crudo,
       cast(egresos  as varchar(100)) as egresos_crudo,
       count(*) as veces
from co_sandbox_datos.{TABLA_PEP}
group by 1, 2
order by veces desc
limit 15;
''')

,ingresos_crudo,egresos_crudo,veces
0,0,0,230452
1,26334000,0,802
2,4000000,2000000,777
3,3000000,2000000,769
4,5000000,3000000,684
5,2000000,1000000,664
6,3000000,1500000,614
7,877803,0,607
8,5000000,2000000,573
9,3000000,1000000,435


In [18]:
# D7 — La regla FILA A FILA del bloque B (rg_consistencia_*) sí está calculada,
#      pero el bloque C NO la usa (usa la razón de sumas). Así se ve su distribución:
run_query('''
select
    coalesce(cast(tipo_persona as varchar), 'NULL') as tipo_persona,
    count(*)                                                                 as clientes,
    sum(case when rg_consistencia_ingresos = 1 then 1 else 0 end)            as ing_consistente,
    sum(case when rg_consistencia_ingresos = 0 then 1 else 0 end)            as ing_inconsistente,
    sum(case when rg_consistencia_ingresos is null then 1 else 0 end)        as ing_sin_base_comparacion,
    sum(case when rg_consistencia_egresos = 1 then 1 else 0 end)             as egr_consistente,
    sum(case when rg_consistencia_egresos = 0 then 1 else 0 end)             as egr_inconsistente,
    sum(case when rg_consistencia_egresos is null then 1 else 0 end)         as egr_sin_base_comparacion
from tmp_reglas_reporte_base
group by 1
order by 1;
''')

,tipo_persona,clientes,ing_consistente,ing_inconsistente,ing_sin_base_comparacion,egr_consistente,egr_inconsistente,egr_sin_base_comparacion
0,1,228603,82231,144722,1650,60975,165978,1650
1,2,28623,11,28445,167,23,28433,167
2,NULL,35,0,0,35,0,0,35


In [19]:
# D8 — Comprobación empírica: ¿cómo trata Redshift los NULL dentro de GREATEST/LEAST?
#      Define si el camino (2) (SUM(pirani) = 0) devolvería NULL o 0.
run_query('''
select greatest(0, cast(null as numeric(18,2)))  as greatest_con_null,
       least(100, cast(null as numeric(18,2)))   as least_con_null,
       1 - (cast(null as numeric(18,2)))         as resta_con_null;
''')

,greatest_con_null,least_con_null,resta_con_null
0,0,100,None


In [20]:
# D9 — Veredicto automático: cruza el embudo con la métrica y concluye.
emb = run_query('''
select tipo_persona,
       count(*) as clientes,
       sum(case when declara_sarlaft = 1 and ingresos_pirani is not null then 1 else 0 end) as universo_ing,
       sum(case when declara_sarlaft = 1 and egresos_pirani  is not null then 1 else 0 end) as universo_egr,
       sum(case when ingresos_pirani is not null then 1 else 0 end) as con_pirani_ing,
       sum(case when declara_sarlaft = 1 then 1 else 0 end) as con_sarlaft
from tmp_reglas_reporte_base
where tipo_persona in (1,2)
group by tipo_persona
order by tipo_persona;
''')

for _, f in emb.iterrows():
    tp = int(f['tipo_persona'])
    etiqueta = 'Persona natural' if tp == 1 else 'Persona juridica'
    print()
    print('=== {} (tipo_persona = {}) ==='.format(etiqueta, tp))
    print('  clientes                            : {:,}'.format(int(f['clientes'])))
    print('  con formulario SARLAFT              : {:,}'.format(int(f['con_sarlaft'])))
    print('  con ingresos en Pirani              : {:,}'.format(int(f['con_pirani_ing'])))
    print('  universo de consistencia (ingresos) : {:,}'.format(int(f['universo_ing'])))
    print('  universo de consistencia (egresos)  : {:,}'.format(int(f['universo_egr'])))
    if int(f['universo_ing']) == 0:
        motivo = ('no cruza con Pirani' if int(f['con_pirani_ing']) == 0
                  else 'cruza con Pirani pero no tiene formulario SARLAFT (declara_sarlaft = 1)')
        print('  --> pct_consistencia = NULL porque el universo esta VACIO: ' + motivo + '.')
    else:
        print('  --> el universo NO esta vacio, asi que pct_consistencia NO puede ser NULL: '
              'si ves 0.00 es truncamiento de la formula (desviacion > 100% o base en cero), '
              'ver D2. Los NULL de consistencia en los OTROS atributos son normales: la '
              'dimension solo se calcula para total_ingresos y total_egresos.')


=== Persona natural (tipo_persona = 1) ===
  clientes                            : 228,603
  con formulario SARLAFT              : 98,034
  con ingresos en Pirani              : 226,953
  universo de consistencia (ingresos) : 97,178
  universo de consistencia (egresos)  : 97,178
  --> el universo NO esta vacio, asi que pct_consistencia NO puede ser NULL: si ves 0.00 es truncamiento de la formula (desviacion > 100% o base en cero), ver D2. Los NULL de consistencia en los OTROS atributos son normales: la dimension solo se calcula para total_ingresos y total_egresos.

=== Persona juridica (tipo_persona = 2) ===
  clientes                            : 28,623
  con formulario SARLAFT              : 525
  con ingresos en Pirani              : 28,456
  universo de consistencia (ingresos) : 522
  universo de consistencia (egresos)  : 522
  --> el universo NO esta vacio, asi que pct_consistencia NO puede ser NULL: si ves 0.00 es truncamiento de la formula (desviacion > 100% o base en cero), ve

## D10–D11 — Causa raíz: por qué casi ninguna empresa tiene `declara_sarlaft = 1`

`declara_sarlaft` se enciende con el join `p.nnumide = s.num_identificacion` (iAxis ↔ formulario
SARLAFT). Ese join es el que define el universo de consistencia. Las dos celdas siguientes miden
si el NIT está escrito igual en ambos lados.

In [21]:
# D10 — Cruce de NIT entre el formulario SARLAFT y iAxis: directo vs. quitando el
#       dígito de verificación al NIT de iAxis.
run_query('''
with iax as (
    select distinct btrim(cast(nnumide as varchar(50))) as doc
    from gde_adp_ods.axis_per_personas
    where ctipide = 37 and nnumide is not null
),
frm as (
    select distinct btrim(cast(num_identificacion as varchar(50))) as doc
    from dp_dm_compliance.vm_formulario_sarlaft
    where par_tipo_id = 37
)
select (select count(*) from iax)                                                        as nits_iaxis,
       (select count(*) from frm)                                                        as nits_formulario,
       (select count(*) from iax join frm on iax.doc = frm.doc)                          as cruce_directo,
       (select count(*) from iax join frm on left(iax.doc, length(iax.doc)-1) = frm.doc) as cruce_iaxis_sin_dv;
''')

,nits_iaxis,nits_formulario,cruce_directo,cruce_iaxis_sin_dv
0,283971,176821,3349,154127


In [22]:
# D11 — Largo del NIT por fuente: iAxis y Pirani lo guardan CON dígito de verificación
#       (10 dígitos); el formulario SARLAFT lo guarda SIN él (9 dígitos).
run_query(f'''
select 'iAxis (axis_per_personas, ctipide=37)' as fuente,
       length(btrim(cast(nnumide as varchar(50)))) as largo, count(*) as documentos
from gde_adp_ods.axis_per_personas where ctipide = 37 group by 1,2
union all
select 'Formulario SARLAFT (par_tipo_id=37)',
       length(btrim(cast(num_identificacion as varchar(50)))), count(*)
from dp_dm_compliance.vm_formulario_sarlaft where par_tipo_id = 37 group by 1,2
union all
select 'Pirani (tipo persona = J)',
       length(btrim(identificationnumber)), count(*)
from co_sandbox_datos.{TABLA_PEP} where "tipo persona" = 'J' group by 1,2
order by fuente, documentos desc;
''')

,fuente,largo,documentos
0,Formulario SARLAFT (par_tipo_id=37),9,159681
1,Formulario SARLAFT (par_tipo_id=37),8,15035
2,Formulario SARLAFT (par_tipo_id=37),7,1980
3,Formulario SARLAFT (par_tipo_id=37),6,90
4,Formulario SARLAFT (par_tipo_id=37),5,15
5,Formulario SARLAFT (par_tipo_id=37),11,13
6,Formulario SARLAFT (par_tipo_id=37),4,3
7,Formulario SARLAFT (par_tipo_id=37),3,1
8,Formulario SARLAFT (par_tipo_id=37),15,1
9,Formulario SARLAFT (par_tipo_id=37),12,1


## Exportar agregados

Solo porcentajes agregados — sin datos personales.

In [23]:
salida = REPO / 'reports'
salida.mkdir(exist_ok=True)

df_metricas.assign(periodo=PERIODO).to_csv(
    salida / 'dq_clientes_sarlaf.csv', index=False, encoding='utf-8-sig')
df_sfc.assign(periodo=PERIODO).to_csv(
    salida / 'dq_clientes_sarlaf_sfc.csv', index=False, encoding='utf-8-sig')

print('Guardados:')
print(' ', salida / 'dq_clientes_sarlaf.csv')
print(' ', salida / 'dq_clientes_sarlaf_sfc.csv')

Guardados:
  C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\dq_clientes_sarlaf.csv
  C:\Users\Wilson.Jerez\OneDrive - HDI Seguros\Documentos\hdi-primas-data-quality\reports\dq_clientes_sarlaf_sfc.csv


# Hallazgos sobre `sarlaf.sql`

Documentados, sin modificar el SQL original. Números de la corrida del periodo **202606** con
`clientes_pirani_202606` (los mismos valores que están quemados en `sarlaf.sql`), ejecutada el
2026-07-24 desde `sarlaf_runner.ipynb` y `auditoria_consistencia.sql`.

## 1. El `0.00` de consistencia en persona jurídica es un artefacto de comparar contra ceros

No es un problema de calidad de las cifras. La evidencia:

- El universo jurídico son **522 clientes**. De ellos **512 tienen `ingresos_pirani = 0`** y solo
  **10** tienen un valor mayor que cero.
- Esos 512 aportan **$2.739.303.453.422 al numerador** y **$0 al denominador**. Solo 3 empresas
  (ingresos > 100.000 M) aportan $2.599.193.472.950, el **87 % del "error"**.
- Donde Pirani sí tiene dato, la cifra **coincide exactamente**: las 10 empresas comparables dan
  razón reporte/Pirani = 1.0000 (mínimo = mediana = máximo) y las 10 son iguales al peso.
  La fórmula aplicada solo a ese subconjunto da **100.00**.
- Egresos es más extremo aún: 519 de 522 con `egresos_pirani = 0`, desviación 129,42 → `0.00`.

**Causa de fondo:** el universo se filtra con `ingresos_pirani IS NOT NULL`, pero en Pirani el
"sin dato" llega como la cadena `'0'`, no como `NULL` (230.452 de 357.140 registros). El
`IS NOT NULL` nunca los excluye, así que la regla interpreta "Pirani afirma que esta empresa
tiene ingresos de $0" cuando en realidad significa "Pirani no capturó el dato".

**Corrección sugerida** (decisión de negocio, no aplicada): cambiar el universo a
`ingresos_pirani > 0` y reportar siempre la **cobertura** junto al porcentaje. Con ese cambio el
indicador daría 100.00 en jurídica y ~99,7 en natural — pero sobre 10 y ~82.800 clientes, es decir
el 0,03 % de las empresas. Publicar ese 100 sin declarar la cobertura sería tan engañoso como el
`0.00` actual.

## 2. La fórmula no tiene resolución fuera del rango 0–100 % de desviación

`LEAST(100, GREATEST(0, (1 − desviación_relativa) × 100))` aplasta a 0 cualquier desviación mayor
al 100 %: un −20 % y un −1090 % se ven idénticos. Además no es un "% de clientes que cumplen" como
las demás dimensiones, sino una razón entre **dos sumas** del portafolio, que se la puede llevar un
solo cliente grande (aquí, 3 empresas hacen el 87 %).

Detalle de fragilidad: si esas columnas dejaran de ser `NUMERIC` y fueran enteras, la división
pasaría a ser **entera** (ratio 11 en vez de 11,903787) y el indicador cambiaría de valor en
silencio. Hoy son `NUMERIC(18,2)` por los `TRY_CAST` del bloque A.

### Variantes evaluadas para acotar el rango sin `GREATEST` (Q8)

Poner más `ABS` no sirve: el `ABS` ya está en el numerador y lo que se dispara es el **cociente**,
que no tiene cota superior. Lo que sí acota por construcción es cambiar el **denominador**.

| variante | fórmula | ingresos natural (97.178) | ingresos jurídica (522) | egresos jurídica (522) |
|---|---|---|---|---|
| V0 actual | `1 − \|a−b\|/b`, truncada | 98,28 | **0,00** (crudo −1.090,37) | **0,00** (crudo −12.842,11) |
| V1 simétrica | `1 − \|a−b\|/(a+b)` | 99,15 | 14,39 | 1,53 |
| V2 min/max | `min(a,b)/max(a,b)` | 98,30 | 7,74 | 0,76 |

Como `a, b ≥ 0`, siempre se cumple `|a−b| ≤ a+b`, así que **V1 está en [0,100] por construcción**,
sin `GREATEST` ni `LEAST`. V2 también está acotada pero usa esas mismas funciones. V2 es casi
idéntica a V0 en el rango sano (98,30 vs 98,28), así que preserva la comparabilidad con lo ya
reportado; V1 sube todo ~0,9 pp.

Ninguna de las dos arregla el problema de los ceros del punto 1: solo cambia la escala con que se
reporta el artefacto.

**V3 — % de clientes que coinciden** (universo `pirani > 0`, tolerancia ±1):

| | clientes | % coinciden |
|---|---|---|
| Ingresos natural | 82.775 | 95,65 |
| Ingresos jurídica | 10 | 100,00 |
| Egresos natural | 70.385 | 77,61 |
| Egresos jurídica | 3 | 66,67 |

Es un `AVG` de 0/1: acotado por construcción, sin división que se dispare, sin que un cliente grande
domine, y comparable con completitud/validez/unicidad. La brecha con la razón de sumas es
reveladora: en egresos de natural, las **sumas** cuadran al 99,82 % pero solo el **77,61 % de los
clientes** tiene la cifra correcta — los errores en un sentido cancelan los del otro y el agregado
los esconde.

> Trampa de Redshift al implementarlo: `AVG(CASE WHEN ... THEN 1.0 ELSE 0.0 END)` trunca el
> resultado a **un decimal** (la escala de `1.0`). Hay que castear a `DECIMAL(18,6)` como ya hace el
> bloque C original. Con `1.0` el % de ingresos natural sale 90,0 en vez de 95,65.

## 3. Dónde el `NULL` de `pct_consistencia` sí es normal

La dimensión consistencia solo está definida para `total_ingresos` y `total_egresos`. En los otros
8 atributos (`numero_documento`, `celular`, …) `pct_consistencia` sale `NULL` para ambos tipos de
persona, y es correcto.

## 4. El NIT del formulario SARLAFT no cruza con el de iAxis

El formulario guarda el NIT **sin dígito de verificación** (9 dígitos: 159.681 de 176.821) e iAxis
**con** DV (10 dígitos: 260.948). El join `p.nnumide = s.num_identificacion` cruza solo **3.349**
NIT; quitándole el DV al lado de iAxis cruzan **154.127** — 46 veces más.

Consecuencia: solo **525 de 28.623** tomadores jurídicos (1,8 %) quedan con `declara_sarlaft = 1`,
contra 42,9 % en personas naturales. Afecta a `clasificacion`, `perfil_riesgo` y a los cuatro
atributos financieros (activos/pasivos/ingresos/egresos), que salen del formulario: toda la
medición de personas jurídicas de esos campos queda sesgada.

*Pirani sí cruza bien* (28.456 de 28.623) porque trae el NIT con DV, igual que iAxis.

**Alcance medido — este bug NO es lo que limita la cobertura de consistencia.** Se comprobó
quitando el filtro `declara_sarlaft = 1` del universo: la cobertura queda **idéntica** (10 clientes
en jurídica, 82.837 en natural) y el porcentaje no se mueve. La razón es que quien tiene cifra en
Pirani ya tiene formulario, así que el filtro no está descartando a nadie. El limitante de
consistencia es el punto 12, no este.

## 12. El techo de cobertura de consistencia lo pone Pirani, no el join

En la tabla `clientes_pirani_202606` (357.140 registros):

| tipo persona en Pirani | filas | con `ingresos > 0` | con `egresos > 0` |
|---|---|---|---|
| J (empresas) | 39.878 | **156** (0,4 %) | **54** (0,1 %) |
| N (naturales) | 317.262 | 125.363 (39,5 %) | 103.488 (32,6 %) |

Pirani prácticamente no trae cifras financieras de empresas. Embudo sobre los tomadores vigentes:

| paso | natural | jurídica |
|---|---|---|
| 1. clientes vigentes | 228.603 | 28.623 |
| 2. cruzan con Pirani | 226.953 | 28.456 |
| 3. Pirani trae ingresos > 0 | 82.837 | **10** |
| 4. (además) con formulario SARLAFT | 98.034 | 525 |
| 5. universo de consistencia v2 | 82.837 | **10** |

El paso 3 es el que manda: el paso 4 no resta nada. Por eso la cobertura de consistencia en
jurídica no puede subir arreglando joins — no existe el dato de contraste. Solo subiría si Pirani
empezara a entregar ingresos/egresos de empresas.

## 13. Los tres números que conviene publicar por campo

Consistencia se diferencia de completitud/validez/unicidad en que necesita una **segunda fuente**.
Por eso un solo porcentaje no alcanza: hay que decir también sobre cuánta población se calculó.

Ya implementado en el bloque C de `sarlaf_v2.sql`. Salida real del periodo 202606:

| tipo | atributo | `pct_consistencia` | `pct_cobertura_consistencia` | `pct_consistencia_ponderada` | `cumple_90` | `alerta_cobertura_baja` |
|---|---|---|---|---|---|---|
| Natural | total_ingresos | 95,78 | 36,24 | 34,71 | 1 | 0 |
| Natural | total_egresos | 78,08 | 30,81 | 24,05 | 0 | 0 |
| Jurídica | total_ingresos | **100,00** | **0,03** | **0,03** | **1** | **1** |
| Jurídica | total_egresos | 66,67 | 0,01 | 0,01 | 0 | **1** |

- **Cobertura** = comparables / total de clientes → qué tan verificable es el campo.
- **Consistencia** = coinciden / comparables → de lo verificable, cuánto cuadra (es lo que hoy
  entrega `sarlaf_v2.sql`).
- **Ponderada** = coinciden / total de clientes → el número sobre el 100 % de la población;
  equivale a consistencia × cobertura y cuenta a los no comparables como "no verificados".

Publicar solo el 100,00 de jurídica es tan engañoso como el `0.00` original: está calculado sobre
10 clientes de 28.623. Con los tres números juntos queda claro de inmediato que el problema es la
fuente, no los datos de HDI.

## 5. `rg_consistencia_ingresos` / `rg_consistencia_egresos` son código muerto

El bloque B las calcula fila a fila (tolerancia ±1), pero los bloques C y SFC no las usan: usan la
razón de sumas. Dan resultados incompatibles — la regla fila a fila da **0,04 %** en jurídica y
**36,23 %** en natural, frente al 98,28 % de la razón de sumas.

Además arrastra el mismo defecto del punto 1: marca `0 = inconsistente` cuando `ingresos_pirani = 0`,
o sea cuando la comparación era imposible.

## 6. El bloque SFC usa `CREATE TABLE` (permanente), no `CREATE TEMP TABLE`

Deja `tmp_reporte_sfc_calidad_datos` en el esquema por defecto y hace fallar la segunda ejecución
("table already exists"). El notebook lo ejecuta como TEMP.

## 7. El bloque C y el bloque SFC no reportan lo mismo

En el SFC están comentadas las filas de oportunidad de `numero_documento`, `nombres_completos`,
`fecha_nacimiento` y `fecha_vinculacion`; en el bloque C sí están. Los dos entregables difieren en
esas 4 celdas.

## 8. `fecha_actualizacion_*` financieras caen a `p.fmovimi_persona` aunque el valor no exista

(`ELSE p.fmovimi_persona` en activos/pasivos/ingresos/egresos.) Un cliente sin ingresos puede
puntuar "oportuno" en ingresos: la oportunidad queda inflada.

## 9. `total_ingresos` para jurídica sale de `s.ingreso_laboral`

En la cascada `COALESCE(sct.total_ingresos_sct, s.ingreso_laboral, d.ingresos)`. "Ingreso laboral"
es un concepto de persona natural; conviene confirmar con negocio qué campo del formulario aplica a
una empresa.

## 10. El parseo de Pirani no está fallando hoy, pero es frágil

`REGEXP_REPLACE(..., '[^0-9.-]', '')` sobre un formato colombiano `1.234.567,89` produciría
`1.234.567.89` y `TRY_CAST` devolvería `NULL` sin error. En `clientes_pirani_202606` los 357.140
registros parsean bien, pero basta un cambio de formato en el cargue para perder valores en silencio.

## 11. El reporte no es reproducible bit a bit entre corridas

Varios `ROW_NUMBER() OVER (... ORDER BY fmovimi DESC)` no tienen desempate determinista: con dos
registros del mismo `sperson` y misma fecha, cada corrida puede elegir uno distinto. Entre dos
ejecuciones seguidas, la consistencia de ingresos de persona natural se movió de 98,28 a 98,29.

> Detalle numérico y las consultas que lo prueban: [`auditoria_consistencia.sql`](auditoria_consistencia.sql) (Q0–Q8). Versión corregida del SQL: [`sarlaf_v2.sql`](sarlaf_v2.sql).

In [24]:
cursor.close()
conn.close()
print('Conexión cerrada.')

Conexión cerrada.
